# P5-2 · GĐ 2 · Diffusion cho ảnh

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q diffusers

## Lý thuyết

In [ ]:
import torch
from diffusers import UNet2DModel
model = UNet2DModel(sample_size=32, in_channels=1, out_channels=1, layers_per_block=2,
                    block_out_channels=(32, 64, 128),
                    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
                    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"))
x_t, t = torch.randn(4, 1, 32, 32), torch.randint(0, 1000, (4,))   # batch 4 ảnh nhiễu và bước t
eps_hat = model(x_t, t).sample          # x_t: (B,1,32,32), t: (B,) số nguyên -> (B,1,32,32)
print(eps_hat.shape, f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} triệu tham số")

torch.Size([4, 1, 32, 32]) 4.02 triệu tham số


In [ ]:
import torch
T = 1000
alpha_bars = torch.cumprod(1 - torch.linspace(1e-4, 0.02, T), dim=0)   # lịch nhiễu như GĐ1

@torch.no_grad()
def sample_ddim(model, n, steps=50):
    x = torch.randn(n, 1, 32, 32)
    ts = torch.linspace(T - 1, 0, steps).round().long().tolist()
    for i, t in enumerate(ts):
        eps = model(x, torch.full((n,), t)).sample
        ab = alpha_bars[t]
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        if i == len(ts) - 1:
            return x0.clamp(-1, 1)
        ab_s = alpha_bars[ts[i + 1]]
        x = ab_s.sqrt() * x0 + (1 - ab_s).sqrt() * eps

imgs = sample_ddim(model.eval(), n=2, steps=10)    # model ở ô trên chưa train nên ảnh chỉ là nhiễu; ở đây chỉ kiểm tra chạy đúng
imgs.shape, imgs.min().item(), imgs.max().item()

(torch.Size([2, 1, 32, 32]), -1.0, 1.0)


In [ ]:
import copy, torch
ema = copy.deepcopy(model).eval().requires_grad_(False)

@torch.no_grad()
def update_ema(ema, model, decay=0.999):
    for pe, p in zip(ema.parameters(), model.parameters()):
        pe.mul_(decay).add_(p, alpha=1 - decay)
# gọi update_ema(ema, model) sau mỗi opt.step(); sinh ảnh bằng ema, không phải model

# thử: đổi trọng số model rồi cập nhật EMA 1 lần và 1000 lần
w = next(model.parameters()); w_ema = next(ema.parameters())
with torch.no_grad(): w.add_(1.0)                     # giả sử một bước train làm trọng số tăng 1
update_ema(ema, model)
print("sau 1 lần:", round((w - w_ema).abs().mean().item(), 4))      # EMA mới đi được 0,1% quãng đường
for _ in range(999): update_ema(ema, model)
print("sau 1000 lần:", round((w - w_ema).abs().mean().item(), 4))   # còn khoảng e^-1 ≈ 37%

sau 1 lần: 0.999
sau 1000 lần: 0.3677


## Bài tập 4: U-Net mini có nhận bước t {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math, torch, torch.nn as nn

def time_embedding(t, dim=32):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half) / half)
    a = t.float().unsqueeze(1) * f.unsqueeze(0)
    return torch.cat([a.sin(), a.cos()], dim=1)          # (B, dim)

class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: down1, pool, down2, t_mlp, up, out
    def forward(self, x, t):
        # TODO: trả về tensor cùng shape với x
        return x

In [ ]:
class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.SiLU())
        self.pool = nn.MaxPool2d(2)
        self.down2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.SiLU())
        self.t_mlp = nn.Linear(32, 32)
        self.up = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.out = nn.Conv2d(32, 1, 3, padding=1)
    def forward(self, x, t):
        h1 = self.down1(x)                                    # (B, 16, 8, 8)
        h2 = self.down2(self.pool(h1))                        # (B, 32, 4, 4)
        h2 = h2 + self.t_mlp(time_embedding(t))[:, :, None, None]
        u = self.up(h2)                                       # (B, 16, 8, 8)
        c = torch.cat([u, h1], dim=1)                         # (B, 32, 8, 8)
        shapes.update(h1=tuple(h1.shape), h2=tuple(h2.shape), concat=tuple(c.shape))
        return self.out(c)

shapes = {}
torch.manual_seed(0)
model = MiniUNet()
y = model(torch.randn(4, 1, 8, 8), torch.tensor([1, 10, 100, 999]))
shapes["out"] = tuple(y.shape)
print(shapes, "| tham số:", sum(p.numel() for p in model.parameters()))

{'h1': (4, 16, 8, 8), 'h2': (4, 32, 4, 4), 'concat': (4, 32, 8, 8), 'out': (4, 1, 8, 8)} | tham số: 8209


## Bài tập 5: DDIM ít bước mà vẫn đúng phân phối {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
MU, SIG, T = 2.0, 0.5, 1000
betas = np.linspace(1e-4, 0.02, T); alphas = 1 - betas; abar = np.cumprod(alphas)

def eps_toi_uu(x, t):
    """Nhiễu dự đoán tốt nhất khi dữ liệu ~ N(MU, SIG²)."""
    a = abar[t]
    return np.sqrt(1 - a) * (x - np.sqrt(a) * MU) / (a * SIG ** 2 + 1 - a)

def ddpm(n, rng):
    x = rng.standard_normal(n)
    for t in reversed(range(T)):
        x = (x - betas[t] / np.sqrt(1 - abar[t]) * eps_toi_uu(x, t)) / np.sqrt(alphas[t])
        if t > 0:
            x += np.sqrt(betas[t]) * rng.standard_normal(n)
    return x

def ddim(n, so_buoc, rng):
    # TODO: các bước t cách đều từ T-1 về 0; mỗi bước: đoán x0 rồi nhảy tới t_kế (η = 0)
    return rng.standard_normal(n)

In [ ]:
def ddim(n, so_buoc, rng):
    ts = np.linspace(T - 1, 0, so_buoc).astype(int)
    x = rng.standard_normal(n)
    for i, t in enumerate(ts):
        e = eps_toi_uu(x, t)
        x0 = (x - np.sqrt(1 - abar[t]) * e) / np.sqrt(abar[t])
        a_s = abar[ts[i + 1]] if i + 1 < len(ts) else 1.0
        x = np.sqrt(a_s) * x0 + np.sqrt(1 - a_s) * e
    return x

kq = {"DDPM 1000": ddpm(20000, np.random.default_rng(0)),
      "DDIM 50": ddim(20000, 50, np.random.default_rng(0)),
      "DDIM 10": ddim(20000, 10, np.random.default_rng(0))}
for ten, x in kq.items():
    print(f"{ten:9s}: trung bình {x.mean():.3f}, độ lệch chuẩn {x.std():.3f}")

DDPM 1000: trung bình 2.004, độ lệch chuẩn 0.499
DDIM 50  : trung bình 1.996, độ lệch chuẩn 0.470
DDIM 10  : trung bình 1.997, độ lệch chuẩn 0.360


## Bài tập 6: Lịch cosine giữ tín hiệu lâu hơn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
T, s = 1000, 0.008
t = np.arange(T)
# TODO: abar_lin, abar_cos; ti_le_tin_hieu[tên], ti_le_nhieu[tên]

In [ ]:
abar_lin = np.cumprod(1 - np.linspace(1e-4, 0.02, T))
f = np.cos(((t + 1) / T + s) / (1 + s) * np.pi / 2) ** 2
abar_cos = f / np.cos(s / (1 + s) * np.pi / 2) ** 2
ti_le_tin_hieu, ti_le_nhieu = {}, {}
fig, ax = plt.subplots(figsize=(6, 3))
for ten, a in [("tuyến tính", abar_lin), ("cosine", abar_cos)]:
    ti_le_tin_hieu[ten] = (a > 0.5).mean(); ti_le_nhieu[ten] = (np.sqrt(a) < 0.1).mean()
    ax.plot(t, np.sqrt(a), label=ten)
    print(f"{ten:10s}: tín hiệu ưu thế {ti_le_tin_hieu[ten]:.1%}, gần như toàn nhiễu {ti_le_nhieu[ten]:.1%}")
ax.set(xlabel="t", ylabel="√ᾱ_t (hệ số tín hiệu)", title="Cosine giảm tín hiệu đều hơn"); ax.legend()

tuyến tính: tín hiệu ưu thế 25.9%, gần như toàn nhiễu 32.7%
cosine    : tín hiệu ưu thế 49.6%, gần như toàn nhiễu 6.5%
